# Overview

In [ ]:
from functools import partial
import numpy as np
from lucifex.fem import average_grid, cross_section_grid
from lucifex.fdm import GridFunctionSeries, NPyConstantSeries
from lucifex.io import write
from lucifex.sim import run, xdmf_to_npz
from lucifex.plt import (
    plot_colormap, create_animation, plot_line, save_figure, plot_twin_lines,
    display_animation, get_ipynb_file_name, set_ipynb_variable, create_multifigure,
)
from lucifex.utils.fenicsx_utils import mesh_axes
from lucifex.utils.npy_utils import as_index, derivative
from crocodil.dns.system_a import dns_system_a, SYSTEM_A_REFERENCE
from crocodil.theory.system_a import (
    threshold_rayleigh, critical_saturation, 
    mass_dissolved_asymptote, mass_capillary_asymptote,
    mass_dissolved_initial, mass_capillary_initial,
)

STORE = 1
WRITE = None
DIR_ROOT = f'./figures/{get_ipynb_file_name()}'
NX = set_ipynb_variable('NX', 60)
NY = set_ipynb_variable('NY', 60)
ANIM = set_ipynb_variable('ANIM', False)

simulation = dns_system_a(
    store_delta=STORE, 
    write_delta=WRITE, 
    dir_root=DIR_ROOT, 
    dir_uid=True,
)(
    Nx=NX,
    Ny=NY,
    scaling='advective',
    **SYSTEM_A_REFERENCE.replace(Ra=800.0),
    dt_max=0.1,
    dt_Cu=0.75,
    dt_Cd=0.75,
    dt_Cr=0.1,
    c_stabilization=None,
    c_limits=True,
    diagnostic=True,
)

Lx, Ly = simulation['Lx', 'Ly']
Ra, Da, epsilon, zeta0, sr, cr = (
    float(i) for i in simulation['Ra', 'Da', 'epsilon', 'zeta0', 'sr', 'cr']
)

Ra_thresh = threshold_rayleigh(Lx, Ly, NX, 2)
print(f"Ra = {Ra} , Ra_thresh = {Ra_thresh}")

sr_crit = critical_saturation(zeta0, cr, epsilon)
print(f'sr = {sr} , sr_crit = {sr_crit}')

save_fig = partial(save_figure, dir_path=simulation.dir_path, prefix=False,)